# Semana 1 — Extract & Load: de Synthea crudo a una capa *bronze* en Parquet

**Objetivo:** descargar un conjunto real de historias clínicas sintéticas (Synthea, MITRE), inspeccionarlo y cargarlo sin modificar en una capa *bronze* en formato Parquet, con metadatos de linaje y una conciliación de filas.

**Producto real usado:** Synthea, muestra pública de ~1.163 pacientes (CSV, noviembre 2021).

**Al terminar tendrás:** `data/bronze/*.parquet` y `data/bronze/_manifest.json` en tu Drive.

**Tiempo estimado:** 2 h 30 min.

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Parámetros de la extracción

**Explicación línea por línea**

- `DATASET_URL = (...)`: dirección del archivo ZIP oficial de Synthea alojado en el repositorio público `synthetichealth/synthea-sample-data` de GitHub. Los paréntesis permiten partir un texto largo en dos líneas; Python las une automáticamente. **Por qué una constante:** si mañana cambias de dataset, cambias una sola línea.
- `RAW_DIR`, `ZIP_PATH`, `EXTRACT_DIR`, `BRONZE_DIR`: rutas de cada zona. `raw` guarda el archivo tal como llegó del proveedor; `bronze` guarda la primera copia estructurada. **En la práctica real** separar zonas permite reprocesar todo desde `raw` si un paso posterior tenía un error.
- `TABLES_IN_SCOPE = [...]`: lista de tablas que sí vamos a cargar. Dejamos fuera `claims`, `claims_transactions` (310 MB) y otras de facturación. **Por qué importa:** cargar solo lo necesario es *minimización de datos*, un principio de privacidad y también de costo computacional.
- `for carpeta in (...): carpeta.mkdir(...)`: recorre una tupla de rutas y crea cada carpeta de forma idempotente.

In [ ]:
DATASET_URL = (
    "https://raw.githubusercontent.com/synthetichealth/synthea-sample-data/"
    "main/downloads/synthea_sample_data_csv_nov2021.zip"
)
RAW_DIR = DATA_DIR / "raw"
ZIP_PATH = RAW_DIR / "synthea_sample_data_csv_nov2021.zip"
EXTRACT_DIR = RAW_DIR / "synthea_csv_nov2021"
BRONZE_DIR = DATA_DIR / "bronze"

TABLES_IN_SCOPE = [
    "patients", "encounters", "conditions", "observations", "medications",
    "procedures", "immunizations", "allergies", "careplans",
    "organizations", "providers",
]

for carpeta in (RAW_DIR, EXTRACT_DIR, BRONZE_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)
print("Tablas en alcance:", len(TABLES_IN_SCOPE))

### Paso 2 — Extract: descarga idempotente con huella SHA-256

**Explicación línea por línea**

- `import hashlib`: módulo estándar de funciones hash criptográficas. `import urllib.request`: cliente HTTP de la librería estándar (no necesitas instalar nada).
- `def sha256_of(path, chunk_size=1024 * 1024):` define una función que calcula la huella SHA-256 de un archivo leyendo bloques de 1 MB.
- `h = hashlib.sha256()`: crea el objeto acumulador del hash.
- `with open(path, "rb") as f:` abre el archivo en modo binario (`rb`); `with` garantiza que se cierre aunque ocurra un error.
- `for chunk in iter(lambda: f.read(chunk_size), b""):` lee bloque a bloque hasta que `read` devuelve `b""` (fin de archivo). **Por qué por bloques:** un archivo de 5 GB no cabe en memoria; así funciona igual con 5 MB o 5 GB.
- `h.update(chunk)`: añade cada bloque al cálculo. `return h.hexdigest()`: devuelve la huella como texto hexadecimal de 64 caracteres.
- `if ZIP_PATH.exists(): ... else: urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)`: solo descarga si el archivo no está. **Por qué importa:** idempotencia; no castigas al servidor del proveedor ni pierdes minutos cada vez que re-ejecutas.
- `zip_sha256 = sha256_of(ZIP_PATH)`: la huella identifica de forma única *esta* versión del archivo. **En la práctica real** se guarda en el manifiesto de ingesta: si el proveedor reemplaza el archivo con el mismo nombre, la huella cambia y lo detectas.
- `ZIP_PATH.stat().st_size / 1e6`: tamaño en megabytes; `:.1f` lo formatea con un decimal.

In [ ]:
import hashlib
import urllib.request


def sha256_of(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


if ZIP_PATH.exists():
    print("Ya descargado; no se vuelve a bajar:", ZIP_PATH.name)
else:
    print("Descargando", DATASET_URL)
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)

zip_sha256 = sha256_of(ZIP_PATH)
print(f"Tamaño: {ZIP_PATH.stat().st_size / 1e6:.1f} MB")
print("SHA-256:", zip_sha256)

### Paso 3 — Inspeccionar el ZIP antes de abrirlo

**Explicación línea por línea**

- `import zipfile`: módulo estándar para leer archivos ZIP sin descomprimirlos todos.
- `with zipfile.ZipFile(ZIP_PATH) as zf:` abre el ZIP en modo lectura.
- `zf.infolist()`: devuelve un objeto por cada archivo interno, con nombre (`filename`) y tamaño descomprimido (`file_size`).
- `[m for m in ... if m.filename.endswith(".csv")]`: *list comprehension* que se queda solo con los CSV (descarta carpetas).
- `print(f"{m.filename:<35} {m.file_size / 1e6:8.2f} MB")`: `:<35` alinea el nombre a la izquierda en 35 caracteres; `:8.2f` alinea el número. **Por qué inspeccionar primero:** en la práctica real un ZIP puede traer 300 MB de facturación que no necesitas (aquí `claims_transactions.csv`). Decidir el alcance antes de cargar ahorra memoria.

In [ ]:
import zipfile

with zipfile.ZipFile(ZIP_PATH) as zf:
    miembros = [m for m in zf.infolist() if m.filename.endswith(".csv")]

for m in miembros:
    print(f"{m.filename:<35} {m.file_size / 1e6:8.2f} MB")
print("CSV dentro del ZIP:", len(miembros))

### Paso 4 — Descomprimir solo las tablas en alcance, de forma segura

**Explicación línea por línea**

- `nombre = Path(m.filename).name`: toma solo el nombre final (`csv/patients.csv` → `patients.csv`). **Por qué importa (seguridad):** un ZIP malicioso puede contener rutas como `../../etc/archivo` (*zip slip*). Al quedarte con `.name` nunca escribes fuera de tu carpeta.
- `Path(nombre).stem`: nombre sin extensión (`patients`), para compararlo con `TABLES_IN_SCOPE`.
- `if ... not in TABLES_IN_SCOPE: continue`: salta las tablas fuera de alcance.
- `if not destino.exists():` idempotencia otra vez: no reescribe lo ya extraído.
- `with zf.open(m) as origen, open(destino, "wb") as salida:` abre dos archivos a la vez: el interno del ZIP para leer y el destino para escribir en binario.
- `shutil.copyfileobj(origen, salida)`: copia por bloques sin cargar todo en memoria.
- `sorted(EXTRACT_DIR.glob("*.csv"))`: lista ordenada de los CSV ya extraídos.

In [ ]:
import shutil

with zipfile.ZipFile(ZIP_PATH) as zf:
    for m in miembros:
        nombre = Path(m.filename).name
        if Path(nombre).stem not in TABLES_IN_SCOPE:
            continue
        destino = EXTRACT_DIR / nombre
        if not destino.exists():
            with zf.open(m) as origen, open(destino, "wb") as salida:
                shutil.copyfileobj(origen, salida)

csv_files = sorted(EXTRACT_DIR.glob("*.csv"))
print([f.name for f in csv_files])

### Paso 5 — Leer una tabla "tal como viene"

**Explicación línea por línea**

- `import pandas as pd`: librería de tablas en memoria; `pd` es el alias universal.
- `pd.read_csv(..., dtype=str, keep_default_na=False)`: lee **todas** las columnas como texto y no convierte celdas vacías en `NaN`. **Por qué así en bronze:** si pandas adivina tipos, un código como `07` puede volverse `7` o una fecha mal formada puede perderse sin aviso. Databricks recomienda guardar la mayoría de los campos de bronze como texto para protegerse de cambios de esquema inesperados. La conversión de tipos se hace después, de forma explícita y auditada (semana 2).
- `patients_raw.shape`: tupla (filas, columnas).
- `patients_raw.columns.tolist()`: nombres de columnas. Observa `SSN`, `DRIVERS`, `PASSPORT`, `FIRST`, `LAST`, `ADDRESS`: son identificadores directos. En Synthea son ficticios, pero tienen la **forma** de datos personales reales; los eliminaremos en la capa *silver* (semana 3).
- `patients_raw.head(3)`: muestra las 3 primeras filas; en Colab se ve como tabla.

In [ ]:
import pandas as pd

patients_raw = pd.read_csv(EXTRACT_DIR / "patients.csv", dtype=str, keep_default_na=False)
print(patients_raw.shape)
print(patients_raw.columns.tolist())
patients_raw.head(3)

### Paso 6 — Load: escribir la capa bronze en Parquet con metadatos de linaje

**Explicación línea por línea**

- `from datetime import datetime, timezone`: fechas con zona horaria.
- `ingested_at = datetime.now(timezone.utc).isoformat(timespec="seconds")`: marca de tiempo de la carga en UTC y formato ISO 8601 (`2026-11-09T01:23:45+00:00`). **Por qué UTC:** los equipos de datos de salud trabajan entre zonas horarias; UTC evita ambigüedad (Brisbane no tiene horario de verano, Sídney sí).
- `manifest = []`: lista donde registraremos qué se cargó.
- `for csv_path in csv_files:` recorre cada tabla extraída.
- `tabla = csv_path.stem`: nombre lógico de la tabla.
- `df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)`: misma lectura cruda del paso 5.
- `df["_source_file"] = csv_path.name` y `df["_ingested_at"] = ingested_at`: columnas técnicas de **linaje**: de qué archivo vino cada fila y cuándo entró. El prefijo `_` las distingue de las columnas del proveedor. **En la práctica real** esto permite responder en una auditoría "¿de dónde salió este dato?".
- `df.to_parquet(destino, index=False)`: guarda en Parquet, un formato columnar comprimido. **Por qué Parquet y no CSV:** guarda el esquema, ocupa mucho menos y las herramientas analíticas (DuckDB, Spark, BigQuery) leen solo las columnas que necesitan.
- `manifest.append({...})`: registra tabla, filas, columnas del proveedor (`df.shape[1] - 2` descuenta las 2 técnicas), huella del CSV de origen y ruta relativa.
- `print(f"{tabla:<15} {len(df):>9,} filas")`: `:>9,` alinea a la derecha con separador de miles.

In [ ]:
from datetime import datetime, timezone

ingested_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
manifest = []

for csv_path in csv_files:
    tabla = csv_path.stem
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    df["_source_file"] = csv_path.name
    df["_ingested_at"] = ingested_at
    destino = BRONZE_DIR / f"{tabla}.parquet"
    df.to_parquet(destino, index=False)
    manifest.append({
        "table": tabla,
        "rows": len(df),
        "source_columns": df.shape[1] - 2,
        "source_sha256": sha256_of(csv_path),
        "bronze_path": str(destino.relative_to(DATA_DIR)),
    })
    print(f"{tabla:<15} {len(df):>9,} filas")

### Paso 7 — Conciliación: ¿llegó todo?

La conciliación de conteos es el control más básico de cualquier ETL: el número de registros que salió del origen debe ser igual al que llegó al destino.

**Explicación línea por línea**

- `def count_csv_records(path):` cuenta registros del CSV con el lector `csv` de la librería estándar, que respeta comillas (un campo de texto puede contener saltos de línea; contar líneas a ciegas daría un número falso).
- `with open(path, newline="", encoding="utf-8") as f:` `newline=""` es lo que exige el módulo `csv` para manejar bien los saltos de línea.
- `sum(1 for _ in csv.reader(f)) - 1`: cuenta filas y resta el encabezado.
- `pd.read_parquet(..., columns=["_source_file"])`: lee **una sola columna** del Parquet; gracias al formato columnar es casi instantáneo.
- `estado = "OK" if ... else "DIFERENCIA"`: compara origen y destino.
- `assert all(...)`: si alguna tabla no cuadra, el notebook se detiene con error. **En la práctica real** un pipeline nunca debe continuar en silencio tras una pérdida de datos.

In [ ]:
import csv


def count_csv_records(path):
    with open(path, newline="", encoding="utf-8") as f:
        return sum(1 for _ in csv.reader(f)) - 1


resultados = []
for item in manifest:
    origen = count_csv_records(EXTRACT_DIR / f"{item['table']}.csv")
    destino = len(pd.read_parquet(DATA_DIR / item["bronze_path"], columns=["_source_file"]))
    estado = "OK" if origen == destino else "DIFERENCIA"
    resultados.append((item["table"], origen, destino, estado))
    print(f"{item['table']:<15} origen={origen:>9,} bronze={destino:>9,} {estado}")

assert all(r[3] == "OK" for r in resultados), "Hay tablas que no cuadran"

### Paso 8 — Guardar el manifiesto de ingesta

**Explicación línea por línea**

- `import json`: serialización a JSON, legible por humanos y por máquinas.
- `manifest_doc = {...}`: documento con fuente, URL, huella del ZIP, fecha de ingesta y la lista de tablas.
- `json.dumps(manifest_doc, indent=2, ensure_ascii=False)`: convierte a texto con sangría; `ensure_ascii=False` conserva tildes.
- `(BRONZE_DIR / "_manifest.json").write_text(..., encoding="utf-8")`: lo escribe junto a la capa bronze. **Por qué importa:** este archivo es tu registro de linaje (*data lineage*). Las semanas siguientes lo leerán para saber qué versión de datos usan.

In [ ]:
import json

manifest_doc = {
    "source": "Synthea sample data (MITRE), CSV nov2021",
    "source_url": DATASET_URL,
    "zip_sha256": zip_sha256,
    "ingested_at": ingested_at,
    "tables": manifest,
}
(BRONZE_DIR / "_manifest.json").write_text(
    json.dumps(manifest_doc, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Manifiesto escrito con", len(manifest), "tablas")

### Paso 9 — Primer perfil rápido de los datos

**Explicación línea por línea**

- `enc = pd.read_parquet(BRONZE_DIR / "encounters.parquet", columns=[...])`: lee solo 3 columnas de encuentros (consultas, urgencias, hospitalizaciones).
- `enc["START"].min()` / `.max()`: como las fechas están en texto ISO 8601 (`2019-03-01T10:00:00Z`), el orden alfabético coincide con el cronológico; sirve para una mirada rápida. La conversión formal a fecha se hace en la semana 2.
- `enc["PATIENT"].nunique()`: pacientes distintos con al menos un encuentro.
- `enc["ENCOUNTERCLASS"].value_counts()`: frecuencia de cada tipo de encuentro (`wellness`, `ambulatory`, `emergency`, `inpatient`...). **Por qué perfilar ya:** entender la forma de los datos antes de transformarlos evita decisiones equivocadas (por ejemplo, olvidar que existen las urgencias).

In [ ]:
enc = pd.read_parquet(BRONZE_DIR / "encounters.parquet", columns=["PATIENT", "START", "ENCOUNTERCLASS"])
print("Rango de fechas:", enc["START"].min(), "→", enc["START"].max())
print("Pacientes con encuentros:", enc["PATIENT"].nunique())
enc["ENCOUNTERCLASS"].value_counts()

### Paso 10 — Inventario de identificadores directos (privacidad desde el diseño)

**Explicación línea por línea**

- `DIRECT_IDENTIFIERS = [...]`: columnas de `patients` que identifican a una persona por sí solas (número de seguro social, licencia, pasaporte, nombres, dirección, coordenadas).
- `presentes = [c for c in DIRECT_IDENTIFIERS if c in patients_raw.columns]`: verifica cuáles existen realmente en esta versión del archivo (las versiones de Synthea cambian de columnas).
- `(BRONZE_DIR / "_pii_inventory.json").write_text(...)`: guarda el inventario. **En la práctica real** los equipos que manejan datos clínicos mantienen un inventario de datos personales por tabla; es la base para decidir qué se elimina o se seudonimiza antes de que un analista vea los datos.

In [ ]:
DIRECT_IDENTIFIERS = [
    "SSN", "DRIVERS", "PASSPORT", "PREFIX", "FIRST", "MIDDLE", "LAST",
    "SUFFIX", "MAIDEN", "ADDRESS", "LAT", "LON",
]
presentes = [c for c in DIRECT_IDENTIFIERS if c in patients_raw.columns]
(BRONZE_DIR / "_pii_inventory.json").write_text(
    json.dumps({"patients": presentes}, indent=2), encoding="utf-8"
)
print("Identificadores directos en patients:", presentes)

### Autoevaluación (responde en el README de la semana)

1. ¿Por qué la capa bronze guarda todo como texto y no convierte tipos?
2. ¿Qué detecta la huella SHA-256 que un simple nombre de archivo no detecta?
3. Si la conciliación del paso 7 diera "DIFERENCIA" en `observations`, ¿cuáles serían tres causas probables?
4. ¿Qué es la idempotencia y en qué celdas de este notebook la aplicaste?

**Commit sugerido:** `feat(s01): ingesta Synthea CSV a bronze Parquet con manifiesto y conciliación`